# Talanta P2 time-of-day bake on the Kaggle GPU (internal, private)

One run bakes one lightmap set (or a part of it, ONLY) of the web_hero_v2 twin at one time of day, in one Blender process, and renders that time's panorama.
Scene: blender/out/frozen_FL.blend of the render pack (assets/render_pack, 2026-10-26: frozen_FK plus the seat re-check stand detail). Same groups, UVs and file layout as the 07:30 set (assets/web_hero_v2_bake).
Order and test: blender/kaggle/BAKE_ON_KAGGLE.md.

How to run:
- Settings: Accelerator GPU T4 x2, Internet on, input the private dataset talanta-render-pack (the 2026-10-26 pack version).
- Set the variables in the first code cell. "Save Version", then "Save & Run All (Commit)", private. The maps stay as the version output (lightmaps/<HHMM>/).
- Download the output, then stop. Do not leave an idle session open.

Traffic in the maps is Visualization. The output is internal only. Do not publish this notebook or its output.

In [ ]:
# Settings of this run
HHMM = "1030"                 # 1030, 1805 or 2000
PRESET = {"1030": "daylight_artistic", "1805": "sunset_artistic", "2000": "night"}[HHMM]
# 2000: the manifest sun "night" (20:30 EAT, venue lights on; the sky is dark from about 19:00, so the light is the 20:00 light)
ONLY = "stadium_shell,stadium_fins,stadium_ornament,roof"   # the GPU test (CPU sub-run 1a); "" bakes every group and writes bake_log.json
SAMPLES = 128
PANORAMA = False              # True: also the 8192 x 4096 panorama of this time (after a full bake)
print(HHMM, PRESET, ONLY or "all groups")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2; free -g | head -2

In [ ]:
# Blender and the pack go to /tmp, so that the version output holds the frames only.
import glob, json, os, shutil, subprocess, time
WORK = "/tmp/talanta"
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz"  # same as the GPU test
BLENDER = f"{WORK}/blender/blender"
t_setup = time.time()
if not os.path.exists(BLENDER):
    os.makedirs(f"{WORK}/blender", exist_ok=True)
    subprocess.run(f"wget -q {BLENDER_URL} -O /tmp/b.tar.xz && tar -xf /tmp/b.tar.xz -C {WORK}/blender --strip-components=1 "
                   "&& rm /tmp/b.tar.xz", shell=True, check=True)
subprocess.run([BLENDER, "-b", "--version"], check=True)

# The pack root holds blender/ and blender/out/frozen_FL.blend (pack 2026-10-26; frozen_FK.blend in pack 7e19362). The input is read-only, so copy it.
PACK = f"{WORK}/pack/talanta_render_pack"
if not os.path.exists(PACK):
    os.makedirs(f"{WORK}/pack", exist_ok=True)
    unpacked = glob.glob("/kaggle/input/**/talanta_render_pack/blender/out/frozen_F*.blend", recursive=True)
    if unpacked:
        shutil.copytree(unpacked[0].rsplit("/blender/", 1)[0], PACK)
    else:
        parts = sorted(q for q in glob.glob("/kaggle/input/**/talanta_render_pack.tar*", recursive=True) if not q.endswith(".sha256"))
        print(parts)
        subprocess.run(f"cat {' '.join(parts)} | tar -x -C {WORK}/pack", shell=True, check=True)
print(os.listdir(PACK))

In [ ]:
# The pack 2026-10-26 (c14cc71) has no app/public/data/manifest.json. render_panorama.py reads features.site_pad_y from it.
# This cell writes the repo copy (commit 1573714) only if the pack lacks the file. A rebuilt pack makes this cell do nothing.
import base64
_manifest = f"{PACK}/app/public/data/manifest.json"
if not os.path.exists(_manifest):
    os.makedirs(os.path.dirname(_manifest), exist_ok=True)
    open(_manifest, "wb").write(base64.b64decode("eyJvcmlnaW5fdXRtIjogWzI0OTU4Ni4zLCA5ODU1NTMxLjI3XSwgImNycyI6ICJFUFNHOjMyNzM3IiwgImdyb3VuZF9lbGV2X20iOiAxODA5LjYsICJvcmlnaW5fbG9ubGF0IjogWzM2Ljc0OTU5NywgLTEuMzA2MDM3XSwgInNvdXJjZXMiOiB7fSwgInRlcnJhaW4iOiBbeyJmaWxlIjogInRlcnJhaW5fc2l0ZS5iaW4iLCAibngiOiAzMDAsICJueiI6IDMwMCwgIngwIjogLTE0OTUuMTY0Njg5OTIwMTczMywgInowIjogLTE1MDAuNzk2NDkxODY1MDY4NywgImR4IjogMTAuMCwgImR6IjogMTAuMCwgIm1pbiI6IC0zOS41ODQxMjE3MDQxMDE1NiwgIm1heCI6IDI5LjYxMDc0NDQ3NjMxODM2fSwgeyJmaWxlIjogInRlcnJhaW5fZmFyLmJpbiIsICJueCI6IDQwMCwgIm56IjogNDAwLCAieDAiOiAtNTk5NS4xNjQ2ODk5MjAxNzMsICJ6MCI6IC02MDAwLjc5NjQ5MTg2NTA2OSwgImR4IjogMzAuMCwgImR6IjogMzAuMCwgIm1pbiI6IC0xMzYuMDM1MzI0MDk2Njc5NywgIm1heCI6IDE0MC41MDcxMjU4NTQ0OTIyfSwgeyJmaWxlIjogInRlcnJhaW5faG9yaXpvbi5iaW4iLCAibngiOiA0MDAsICJueiI6IDQwMCwgIngwIjogLTE5OTk1LjE2NDY4OTkyMDE3MywgInowIjogLTIwMDAwLjc5NjQ5MTg2NTA3LCAiZHgiOiAxMDAuMCwgImR6IjogMTAwLjAsICJtaW4iOiAtMjg2LjAzODgxODM1OTM3NSwgIm1heCI6IDYwOS40ODk4NjgxNjQwNjI1fV0sICJncm91bmQiOiBbeyJmaWxlIjogImdyb3VuZF9zaXRlLmpwZyIsICJ4MCI6IC0xNTAwLjAsICJ6MCI6IC0xNTAwLjAsICJzaXplIjogMzAwMH0sIHsiZmlsZSI6ICJncm91bmRfZmFyLmpwZyIsICJ4MCI6IC02MDAwLjAsICJ6MCI6IC02MDAwLjAsICJzaXplIjogMTIwMDB9LCB7ImZpbGUiOiAiZ3JvdW5kX2hvcml6b24uanBnIiwgIngwIjogLTIwMDAwLjAsICJ6MCI6IC0yMDAwMC4wLCAic2l6ZSI6IDQwMDAwfV0sICJidWlsZGluZ3MiOiB7InBvcyI6ICJidWlsZGluZ3NfcG9zLmJpbiIsICJjb2wiOiAiYnVpbGRpbmdzX2NvbC5iaW4iLCAiaWR4IjogImJ1aWxkaW5nc19pZHguYmluIiwgInNyYyI6ICJidWlsZGluZ3Nfc3JjLmJpbiIsICJzcmNfY29kZXMiOiBbIm9wZW5fYnVpbGRpbmdzXzIwMjMiLCAib3NtX2Zsb29ycyIsICJkZWZhdWx0X2J5X2FyZWEiXSwgImhlaWdodF9zcmMiOiB7Im9wZW5fYnVpbGRpbmdzXzIwMjMiOiA3NTUwLCAib3NtX2Zsb29ycyI6IDcsICJkZWZhdWx0X2J5X2FyZWEiOiA2Njc3fSwgImNvdW50IjogMTQyMzR9LCAicm9hZHMiOiB7InBvcyI6ICJyb2Fkc19wb3MuYmluIiwgImlkeCI6ICJyb2Fkc19pZHguYmluIiwgInciOiAicm9hZHNfdy5iaW4ifSwgInRyZWVzIjogeyJmaWxlIjogInRyZWVzLmJpbiIsICJjb3VudCI6IDU1Mzk4fSwgImZlYXR1cmVzIjogeyJib3dsX291dGVyIjogW1stOTcuNSwgMjcuOF0sIFstOTMuNjcsIDYwLjMyXSwgWy04Ni4yOSwgODIuNDhdLCBbLTc0Ljg3LCAxMDAuMV0sIFstNjEuNDEsIDEwOS40MV0sIFstNDIuNjQsIDExOS40OF0sIFstMTUuMjQsIDEyNS41XSwgWzIzLjMxLCAxMjQuNDZdLCBbNDguMTYsIDEyMC40MV0sIFs2OC40NCwgMTEzLjA4XSwgWzc4LjMyLCAxMDcuMDJdLCBbODguNzEsIDk2LjE4XSwgWzk4LjM0LCA4MC44XSwgWzEwMy44NywgMjkuODldLCBbMTA1LjYzLCA3LjIxXSwgWzEwNS44NiwgLTE2LjQ3XSwgWzEwMi41MiwgLTYwLjMyXSwgWzk5Ljk3LCAtNzMuMTZdLCBbOTUuMTQsIC04Ni43Nl0sIFs4MS4xOCwgLTEwMy42NF0sIFs2MS44OSwgLTExNi43M10sIFs0Mi42MSwgLTEyNS4yN10sIFsyMi4zNywgLTEyOS42OF0sIFswLjUxLCAtMTMwLjc4XSwgWy0yNi4xMiwgLTEyNy40OF0sIFstNjAuMSwgLTExNy4xMl0sIFstODQuNjcsIC05MC4xM10sIFstOTEuNSwgLTc2LjAyXSwgWy05Ni4wNSwgLTU4LjYyXSwgWy05Ny41NCwgLTE3LjNdLCBbLTk3LjUsIDI3LjhdXSwgImJvd2xfaG9sZSI6IFtbLTUzLjg2LCA0Ny45M10sIFstNTUuOTMsIDQuMzRdLCBbLTUzLjkzLCAtMzMuNzJdLCBbLTUxLjkzLCAtNTcuOTFdLCBbLTQ4LjY0LCAtNjYuOTldLCBbLTQxLjgsIC03NC4wNV0sIFstMjguMzYsIC03OC4zNF0sIFstMTMuOTEsIC04MC44OF0sIFszLjA5LCAtODEuOV0sIFszNS44LCAtNzkuNDFdLCBbNDUuMTksIC03NS44OV0sIFs1My41NywgLTY5LjA5XSwgWzU4LjE1LCAtNTUuNDldLCBbNTguNjgsIC0yMi40OF0sIFs1OS4yMiwgNC45OF0sIFs1OS4xLCAxMC4yOF0sIFs1NS4xNiwgMTAuMTVdLCBbNTQuNjcsIDMwLjddLCBbNTEuNjUsIDU5LjkyXSwgWzQ4LjM2LCA2Ny4yNF0sIFs0MS41MiwgNzMuMDRdLCBbMjEuOTksIDc2Ljg0XSwgWzEuNzEsIDc4LjEyXSwgWy0zOS42MywgNzYuMTNdLCBbLTQ4Ljc3LCA2Ny4wN10sIFstNTMuODYsIDQ3LjkzXV0sICJwaXRjaF96b25lIjogW1stMzUuMzcsIC01OC4zMV0sIFstMzguOTEsIDU2LjkyXSwgWzM2LjgxLCA1Ny4wN10sIFszNy43NSwgLTU3LjBdLCBbLTM1LjM3LCAtNTguMzFdXSwgInBpdGNoX2JlYXJpbmdfZGVnIjogMS43NiwgInNpdGUiOiBbWy0zNzcuOTcsIC0xMzcuMjhdLCBbLTMwNC41NywgNjUuNjZdLCBbLTE4NS45LCAzMjEuOV0sIFs0ODQuNTksIDEwMi40N10sIFsxODcuNjEsIC0zNDkuMjldLCBbLTY1Ljk4LCAtMjU1LjQ5XSwgWy0xNjYuMjEsIC0yMjcuMjhdLCBbLTI1OS4xMSwgLTE4Ni44NV0sIFstMzc3Ljk3LCAtMTM3LjI4XV0sICJzcG9ydCI6IFt7ImtpbmQiOiAicGl0Y2giLCAicmluZyI6IFtbMTM1Ljg0LCAtMTY1LjU3XSwgWzEzNi40OCwgLTY2LjE5XSwgWzE0MC45LCAtNTcuNDRdLCBbMTQ4LjQ4LCAtNDguNTddLCBbMTU0LjY4LCAtNDMuNzhdLCBbMTYxLjk5LCAtNDEuNDZdLCBbMTcxLjYyLCAtNDAuMjRdLCBbMTc5LjE5LCAtNDEuMDddLCBbMTg4LjQzLCAtNDQuMzZdLCBbMTk1LjI5LCAtNDkuM10sIFsyMDAuOTQsIC01NC42Ml0sIFsyMDQuMjUsIC01OS42OV0sIFsyMDYuNzIsIC02NC43NV0sIFsyMDcuOTYsIC02OS42OF0sIFsyMDguOTYsIC0xNjQuNDddLCBbMjA3LjE3LCAtMTcxLjQ0XSwgWzIwMi4zNCwgLTE3OC41Nl0sIFsxOTMuNjYsIC0xODcuNDNdLCBbMTgzLjg5LCAtMTkxLjhdLCBbMTczLjgyLCAtMTk0LjEyXSwgWzE2NS40MywgLTE5My43MV0sIFsxNTYuNzYsIC0xOTAuNTVdLCBbMTQ4LjUsIC0xODQuNjZdLCBbMTQyLjE3LCAtMTc4Ljc3XSwgWzEzOC4xOCwgLTE3My4xN10sIFsxMzUuODQsIC0xNjUuNTddXSwgImhvbGVzIjogW10sICJhcmVhIjogMTAwMjB9LCB7ImtpbmQiOiAidHJhY2siLCAicmluZyI6IFtbMTMzLjcyLCAtMTgxLjE2XSwgWzEzMy43LCAtMTgxLjA5XSwgWzEyNC45OSwgLTE4MS4wM10sIFsxMjUuNzksIC01OC4wN10sIFsxMzIuNjMsIC01OC4xMV0sIFsxMzYuODEsIC00OS44M10sIFsxNDUuNCwgLTM5Ljc0XSwgWzE1Mi40NCwgLTM0LjMyXSwgWzE2MC43NCwgLTMxLjY4XSwgWzE3MS42NywgLTMwLjI5XSwgWzE4MC4yNywgLTMxLjI0XSwgWzE5MC43NCwgLTM0Ljk3XSwgWzE5OC41NCwgLTQwLjU3XSwgWzIwNC45NSwgLTQ2LjYzXSwgWzIwOC42OSwgLTUyLjM3XSwgWzIxMS41MSwgLTU4LjExXSwgWzIxMi45LCAtNjMuNzFdLCBbMjE0LjA1LCAtMTcxLjI5XSwgWzIxMi4wMiwgLTE3OS4yMV0sIFsyMDYuNTQsIC0xODcuMjhdLCBbMTk2LjY5LCAtMTk3LjM2XSwgWzE4NS41OSwgLTIwMi4zMV0sIFsxNzQuMTcsIC0yMDQuOTRdLCBbMTY0LjY0LCAtMjA0LjQ3XSwgWzE1NC44LCAtMjAwLjg5XSwgWzE0NS40MywgLTE5NC4yXSwgWzEzOC4yNSwgLTE4Ny41M10sIFsxMzMuNzIsIC0xODEuMTZdXSwgImhvbGVzIjogW1tbMTM4LjE4LCAtMTczLjE3XSwgWzE0Mi4xNywgLTE3OC43N10sIFsxNDguNSwgLTE4NC42Nl0sIFsxNTYuNzYsIC0xOTAuNTVdLCBbMTY1LjQzLCAtMTkzLjcxXSwgWzE3My44MiwgLTE5NC4xMl0sIFsxODMuODksIC0xOTEuOF0sIFsxOTMuNjYsIC0xODcuNDNdLCBbMjAyLjM0LCAtMTc4LjU2XSwgWzIwNy4xNywgLTE3MS40NF0sIFsyMDguOTYsIC0xNjQuNDddLCBbMjA3Ljk2LCAtNjkuNjhdLCBbMjA2LjcyLCAtNjQuNzVdLCBbMjA0LjI1LCAtNTkuNjldLCBbMjAwLjk0LCAtNTQuNjJdLCBbMTk1LjI5LCAtNDkuM10sIFsxODguNDMsIC00NC4zNl0sIFsxNzkuMTksIC00MS4wN10sIFsxNzEuNjIsIC00MC4yNF0sIFsxNjEuOTksIC00MS40Nl0sIFsxNTQuNjgsIC00My43OF0sIFsxNDguNDgsIC00OC41N10sIFsxNDAuOSwgLTU3LjQ0XSwgWzEzNi40OCwgLTY2LjE5XSwgWzEzNS44NCwgLTE2NS41N10sIFsxMzguMTgsIC0xNzMuMTddXV0sICJhcmVhIjogMzYzOH0sIHsia2luZCI6ICJwaXRjaCIsICJyaW5nIjogW1sxNDkuNTMsIDMxLjExXSwgWzE1MS4xNiwgMTU2LjA5XSwgWzIyNi4yMywgMTU3LjU0XSwgWzIyNy42NCwgMzIuMDZdLCBbMTQ5LjUzLCAzMS4xMV1dLCAiaG9sZXMiOiBbXSwgImFyZWEiOiA5NTkyfV0sICJzaXRlX3BhZF95IjogNi4xMn0sICJsYWJlbHMiOiBbeyJ0ZXh0IjogIkNlbWV0ZXJ5IFJvYWQiLCAieCI6IDI4Mi4yLCAieiI6IDQxOS4yLCAieSI6IDIuNDEyNTY4NzA4NTM3OTQ3LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJDaXR5IE9mIE5haXJvYmkgRHJpdmUiLCAieCI6IDE0ODIuMywgInoiOiA4NC4zLCAieSI6IC04LjkwMDI5NTg1NjYyMjI3OSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiSGF0aGVydSBSb2FkIiwgIngiOiA5NTMuNSwgInoiOiAtMTU4NS4zLCAieSI6IC0xMS43NzUxNzI2MjMzNjc3OCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiSG9tZXdhcmQgQm91bmQiLCAieCI6IC0yODguMiwgInoiOiAxNjU0LjksICJ5IjogMTAuMTk4MjQ0MDkyNzU4NjE0LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJKYW1odXJpIFBhcmsgUm9hZCIsICJ4IjogNDU2LjksICJ6IjogLTExOS40LCAieSI6IC04LjEyNjI4MTc3NjMyOTUwNiwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiSmFtdWh1cmkgUm9hZCIsICJ4IjogNTg1LjAsICJ6IjogMzI2LjcsICJ5IjogLTAuOTMyNTM3MzIwNDA0NDM1NCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiS2FiaXJpYSBSb2FkIiwgIngiOiAtMTQ0Ni43LCAieiI6IC0xNDY3LjgsICJ5IjogMTMuODU2OTA4MTA0MDU3MzExLCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJLYXJhbmRpbmkgUm9hZCIsICJ4IjogODMyLjMsICJ6IjogLTY5Mi41LCAieSI6IC0zLjY1NTk1MTc0NTM4NTIwOTUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIktlbnlhdHRhIERyaXZlIiwgIngiOiAxNTk0LjAsICJ6IjogMTYyLjAsICJ5IjogLTkuNzY4MDc2MjQyMTQ3MTE3LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJLaW5nYXJhIENsb3NlIiwgIngiOiAxMjEzLjIsICJ6IjogLTExMzcuMywgInkiOiAtMjguNTA2OTc3OTcyOTg3NjcsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIktpbmdhcmEgUm9hZCIsICJ4IjogMTMwOS4xLCAieiI6IC0xMTM1LjUsICJ5IjogLTMxLjI2NDMzODk3MzE2MjY5NCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiS2lueWFuanVpIENyZXNjZW50IiwgIngiOiAtODY3LjAsICJ6IjogLTEzNzIuMywgInkiOiA2LjIxODg3NDY0NDIwMjk3LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJLaW55YW5qdWkgUm9hZCIsICJ4IjogLTg1OC4xLCAieiI6IC0xMjE3LjQsICJ5IjogNS44NzI0NjI4OTMzMzY2NiwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTWFjaGFyaWEgUm9hZCIsICJ4IjogMjk3LjQsICJ6IjogLTE2NjEuNSwgInkiOiAtMjAuNjgyMjA3OTg4MjY4Mzc0LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJNYW1hIFdhaHUgUm9hZCIsICJ4IjogLTEwNy43LCAieiI6IC01NzguOSwgInkiOiA1LjMwMjA2MjExNDY2ODE2MSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTWJhYXppIEF2ZW51ZSIsICJ4IjogMTQ3NC4yLCAieiI6IC0xNDgzLjIsICJ5IjogLTEwLjc1ODMzNzA1MzQ1MzcxNSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTWVtb3JpYWwiLCAieCI6IC0zMDcuNywgInoiOiA4Mi42LCAieSI6IDguMjQzNjg4NDY3OTEzMjE2LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJNb2kgRHJpdmUiLCAieCI6IDE1MjYuMSwgInoiOiAzMDYuOCwgInkiOiAtMTEuNTY4MjUxNzM4MTE3NjI0LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJNdC4gRWxnb24gRHJpdmUiLCAieCI6IDE1NDUuOCwgInoiOiAzNjcuNCwgInkiOiAtMTEuNzI4ODI3MjczOTU5OTUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIk5haXZhc2hhIFJvYWQiLCAieCI6IDQxOS4zLCAieiI6IC0xMzk5LjcsICJ5IjogLTIyLjMxMzA0Mzc3ODAxNDg0NCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTmdvbmcgUm9hZCIsICJ4IjogLTYwMS41LCAieiI6IC05Mi4zLCAieSI6IDkuNzEwNTI2ODQ0NTQyMywgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTmdvdGhvIFJvYWQiLCAieCI6IC01NDcuMiwgInoiOiAtMTM0My45LCAieSI6IC02LjgwODM1OTgyNjQyMDg4MywgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTnlha2lueXVhIFJvYWQiLCAieCI6IC01NTguMywgInoiOiAtMTU3OC41LCAieSI6IC02Ljg4Njg5OTczMjIwNDQ4MywgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiT2F1IFJvYWQiLCAieCI6IDEzODAuMywgInoiOiAxLjIsICJ5IjogLTEzLjE1ODYzNjIzNTg4NDYxNSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiUmlhcmEgUm9hZCIsICJ4IjogMTI0NC4zLCAieiI6IC05OTEuMCwgInkiOiAtMTIuMjIyMDE3NTA2ODcwMDg4LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJTb3V0aGVybiBCeXBhc3MiLCAieCI6IC0xNzI3LjYsICJ6IjogMTQxMi4wLCAieSI6IDMwLjQ4OTEyNDE3NDc5NTk2LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJTdC4gRGFuaWVsIENvbWJvbmkgUm9hZCIsICJ4IjogNDgxLjIsICJ6IjogLTY5NS42LCAieSI6IC0xLjI4NjI1MDI3MTkyOTAyMDUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIldhbWJpcmkgUm9hZCIsICJ4IjogLTEzMjEuNSwgInoiOiAtMTc0LjMsICJ5IjogMTYuMzE1MDE5NjIyMTUyMjUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIldhbnllZSBSb2FkIiwgIngiOiAyMzMuNywgInoiOiAtODIwLjYsICJ5IjogMC43MDQ1MTcyMDE2MTc1OTExLCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJOZ29uZyBSb2FkIEZvcmVzdCBTYW5jdHVhcnkiLCAieCI6IC04NDUsICJ6IjogNjYwLCAieSI6IDIzLCAia2luZCI6ICJhcmVhIiwgImVsZXYiOiAxODE3fSwgeyJ0ZXh0IjogIk5nb25nIFJhY2Vjb3Vyc2UiLCAieCI6IC0xNzM2LCAieiI6IDYwNiwgInkiOiA0MSwgImtpbmQiOiAiYXJlYSIsICJlbGV2IjogMTgzNn0sIHsidGV4dCI6ICJLaWJlcmEiLCAieCI6IDM3MjAsICJ6IjogNzY3LCAieSI6IC01OSwgImtpbmQiOiAiYXJlYSIsICJlbGV2IjogMTczNX0sIHsidGV4dCI6ICJKYW1odXJpIFBhcmsiLCAieCI6IDE3MTUsICJ6IjogMTYwLCAieSI6IC0yLCAia2luZCI6ICJhcmVhIiwgImVsZXYiOiAxNzkyfSwgeyJ0ZXh0IjogIktpbGltYW5pIiwgIngiOiAzODI5LCAieiI6IC0xODg4LCAieSI6IC0zNCwgImtpbmQiOiAiYXJlYSIsICJlbGV2IjogMTc2MX0sIHsidGV4dCI6ICJOYWlyb2JpIENCRCIsICJ4IjogODA0OCwgInoiOiAtMjE3OSwgInkiOiAtMTIxLCAia2luZCI6ICJmYXIiLCAiZWxldiI6IDE2NzR9LCB7InRleHQiOiAiTmdvbmcgSGlsbHMgXHUwMGI3IExhbXdpYSIsICJ4IjogLTEyMzQ2LCAieiI6IDEyNjUyLCAieSI6IDY0NiwgImtpbmQiOiAicGVhayIsICJlbGV2IjogMjQ0MX0sIHsidGV4dCI6ICJOZ29uZyBIaWxscyIsICJ4IjogLTEyNDQ4LCAieiI6IDEwMDUyLCAieSI6IDU4OCwgImtpbmQiOiAicGVhayIsICJlbGV2IjogMjM4M30sIHsidGV4dCI6ICJOZ29uZyIsICJ4IjogLTk3NDgsICJ6IjogNTMxNSwgInkiOiAxMzQsICJraW5kIjogImZhciIsICJlbGV2IjogMTkyOX0sIHsidGV4dCI6ICJXaWxzb24gQWlycG9ydCIsICJ4IjogNzI2MSwgInoiOiAxNzI2LCAieSI6IC0xMTcsICJraW5kIjogImZhciIsICJlbGV2IjogMTY3N31dLCAidGltZWxpbmUiOiBbeyJkYXRlIjogIjIwMjMtMDEtMDgiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMCwgInJpbmdfbWVhbl92aXMiOiA4MDd9LCB7ImRhdGUiOiAiMjAyMy0wMS0xMyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDkzM30sIHsiZGF0ZSI6ICIyMDIzLTAxLTE4IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTA0MH0sIHsiZGF0ZSI6ICIyMDIzLTAxLTI4IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE0N30sIHsiZGF0ZSI6ICIyMDIzLTAyLTA3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTM3OH0sIHsiZGF0ZSI6ICIyMDIzLTAyLTEyIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTQ3OX0sIHsiZGF0ZSI6ICIyMDIzLTAyLTE3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTYzMX0sIHsiZGF0ZSI6ICIyMDIzLTAyLTIyIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTU3OH0sIHsiZGF0ZSI6ICIyMDIzLTAzLTA5IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTUxOX0sIHsiZGF0ZSI6ICIyMDIzLTA4LTIxIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE1NX0sIHsiZGF0ZSI6ICIyMDIzLTA4LTI2IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTAwNH0sIHsiZGF0ZSI6ICIyMDIzLTA5LTI1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogODY5fSwgeyJkYXRlIjogIjIwMjMtMTEtMDkiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMCwgInJpbmdfbWVhbl92aXMiOiA3MTB9LCB7ImRhdGUiOiAiMjAyMy0xMS0yOSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDY0M30sIHsiZGF0ZSI6ICIyMDI0LTAxLTIzIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTAzM30sIHsiZGF0ZSI6ICIyMDI0LTAyLTE3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjE5OCwgInJpbmdfbWVhbl92aXMiOiAxNTk2fSwgeyJkYXRlIjogIjIwMjQtMDMtMDMiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMCwgInJpbmdfbWVhbl92aXMiOiA4ODZ9LCB7ImRhdGUiOiAiMjAyNC0wMy0xMyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyNzh9LCB7ImRhdGUiOiAiMjAyNC0wNC0yMiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyODB9LCB7ImRhdGUiOiAiMjAyNC0wNS0yNyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEzMjJ9LCB7ImRhdGUiOiAiMjAyNC0wOC0wNSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEwNjl9LCB7ImRhdGUiOiAiMjAyNC0wOS0wOSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDExOTB9LCB7ImRhdGUiOiAiMjAyNC0wOS0xNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyNDl9LCB7ImRhdGUiOiAiMjAyNC0wOS0xOSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyMzR9LCB7ImRhdGUiOiAiMjAyNC0xMC0wNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wMjQsICJyaW5nX21lYW5fdmlzIjogMTM5OH0sIHsiZGF0ZSI6ICIyMDI0LTEwLTE0IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTI1Mn0sIHsiZGF0ZSI6ICIyMDI0LTEwLTI0IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAwMiwgInJpbmdfbWVhbl92aXMiOiAxMzEzfSwgeyJkYXRlIjogIjIwMjQtMTAtMjkiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMDE0LCAicmluZ19tZWFuX3ZpcyI6IDEyNzR9LCB7ImRhdGUiOiAiMjAyNC0xMi0wOCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDExNzd9LCB7ImRhdGUiOiAiMjAyNC0xMi0wOCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wMDIsICJyaW5nX21lYW5fdmlzIjogMTE4N30sIHsiZGF0ZSI6ICIyMDI0LTEyLTEzIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTIxNX0sIHsiZGF0ZSI6ICIyMDI0LTEyLTI4IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE2M30sIHsiZGF0ZSI6ICIyMDI1LTAxLTA3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE1NH0sIHsiZGF0ZSI6ICIyMDI1LTAxLTIyIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAxLCAicmluZ19tZWFuX3ZpcyI6IDEyNTB9LCB7ImRhdGUiOiAiMjAyNS0wMi0wNiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDExODV9LCB7ImRhdGUiOiAiMjAyNS0wMi0yMSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyNjB9LCB7ImRhdGUiOiAiMjAyNS0wMi0yNiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyODV9LCB7ImRhdGUiOiAiMjAyNS0wMy0wMyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wMDUsICJyaW5nX21lYW5fdmlzIjogMTI3N30sIHsiZGF0ZSI6ICIyMDI1LTA0LTA5IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAyMSwgInJpbmdfbWVhbl92aXMiOiAxMDk0fSwgeyJkYXRlIjogIjIwMjUtMDQtMTciLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMDYsICJyaW5nX21lYW5fdmlzIjogMTM3MH0sIHsiZGF0ZSI6ICIyMDI1LTA3LTI2IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjA1NSwgInJpbmdfbWVhbl92aXMiOiAxNTgxfSwgeyJkYXRlIjogIjIwMjUtMDktMDYiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMDIxLCAicmluZ19tZWFuX3ZpcyI6IDE3OTd9LCB7ImRhdGUiOiAiMjAyNS0wOS0xNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wOTgsICJyaW5nX21lYW5fdmlzIjogMTkzOH0sIHsiZGF0ZSI6ICIyMDI1LTA5LTE2IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAwMiwgInJpbmdfbWVhbl92aXMiOiAxNzk0fSwgeyJkYXRlIjogIjIwMjUtMDktMjQiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMjA1LCAicmluZ19tZWFuX3ZpcyI6IDIxMDN9LCB7ImRhdGUiOiAiMjAyNS0xMS0xNSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC40NDYsICJyaW5nX21lYW5fdmlzIjogMjM5Nn0sIHsiZGF0ZSI6ICIyMDI1LTEyLTA1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjQxNSwgInJpbmdfbWVhbl92aXMiOiAyMzY1fSwgeyJkYXRlIjogIjIwMjUtMTItMjUiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuNDUzLCAicmluZ19tZWFuX3ZpcyI6IDIzNDh9LCB7ImRhdGUiOiAiMjAyNi0wMS0wMiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC40MDEsICJyaW5nX21lYW5fdmlzIjogMjI4NX0sIHsiZGF0ZSI6ICIyMDI2LTAxLTA0IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjQ1MSwgInJpbmdfbWVhbl92aXMiOiAyMzg5fSwgeyJkYXRlIjogIjIwMjYtMDEtMDciLCAicmluZ19icmlnaHRfZnJhYyI6IDAuNjA2LCAicmluZ19tZWFuX3ZpcyI6IDI3MTd9LCB7ImRhdGUiOiAiMjAyNi0wMi0xMSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC44NjIsICJyaW5nX21lYW5fdmlzIjogNDgyMn0sIHsiZGF0ZSI6ICIyMDI2LTA4LTA1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjg5LCAicmluZ19tZWFuX3ZpcyI6IDQ5MzR9LCB7ImRhdGUiOiAiMjAyNi0wOC0xMiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC44ODUsICJyaW5nX21lYW5fdmlzIjogNTEyMn0sIHsiZGF0ZSI6ICIyMDI2LTA4LTI1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjkwNywgInJpbmdfbWVhbl92aXMiOiA1NDI3fSwgeyJkYXRlIjogIjIwMjYtMDktMDQiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuOTE5LCAicmluZ19tZWFuX3ZpcyI6IDU4NTJ9LCB7ImRhdGUiOiAiMjAyNi0wOS0yNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC45MDcsICJyaW5nX21lYW5fdmlzIjogNjA3MH0sIHsiZGF0ZSI6ICIyMDI2LTEwLTAxIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjkwNSwgInJpbmdfbWVhbl92aXMiOiA0OTgwfV19"))
    print("manifest.json written from the notebook copy")
print("site_pad_y", json.load(open(_manifest))["features"]["site_pad_y"])


In [ ]:
# The inline gpu_pre.py of the GPU test notebook (same as blender/kaggle/gpu_pre.py in the repo).
GPU_PRE = f"{WORK}/gpu_pre.py"
open(GPU_PRE, "w").write('''prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for kind in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = kind
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == kind for d in prefs.devices):
        chosen = kind
        break
if chosen is None:
    raise SystemExit("gpu_pre.py: no OptiX or CUDA device found")
for d in prefs.devices:
    d.use = d.type == chosen
s.cycles.device = "GPU"
print("gpu_pre.py:", chosen, [d.name for d in prefs.devices if d.use])
''')
print("setup", round(time.time() - t_setup), "s")

In [ ]:
# The bake: one Blender process; maps, stages.json (light scale and seconds per batch) and, for a full run, bake_log.json
# and talanta_web_baked.blend go to /kaggle/working/lightmaps/<HHMM>/ (the version output)
import threading
OUT = f"/kaggle/working/lightmaps/{HHMM}"
os.makedirs(f"{OUT}/maps", exist_ok=True)
env = dict(os.environ, TALANTA_SITE_BLEND=f"{PACK}/blender/out/frozen_FL.blend", TALANTA_PRE=GPU_PRE,
           TALANTA_BAKE_OUT=OUT, TALANTA_BAKE_PRESET=PRESET, PYTHONUNBUFFERED="1")
gpu_mem_max = {}
def _watch():
    while not _stop.is_set():
        q = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout
        for line in q.strip().splitlines():
            i, used, total = [x.strip() for x in line.split(",")]
            gpu_mem_max[i] = [max(int(used), gpu_mem_max.get(i, [0])[0]), int(total)]
        _stop.wait(10)
_stop = threading.Event(); threading.Thread(target=_watch, daemon=True).start()
args = ([f"only={ONLY}"] if ONLY else []) + [str(SAMPLES)]
cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1", "--python", f"{PACK}/blender/web_hero/bake_lightmaps.py", "--", *args]
print(" ".join(cmd)); t0 = time.time()
try:
    r = subprocess.run(cmd, cwd=PACK, env=env, capture_output=True, text=True)
finally:
    _stop.set()
open(f"{OUT}/bake_run.log", "w").write(r.stdout[-200000:] + r.stderr[-50000:])
print("\n".join(l for l in r.stdout.splitlines() if l.startswith(("bake ", "light scales", "gpu_pre"))))
print("bake", round((time.time() - t0) / 3600, 2), "h, rc", r.returncode)
assert r.returncode == 0, "bake failed: see bake_run.log in the output"

In [ ]:
# The panorama of this time (only after a full bake)
if PANORAMA:
    cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1", "--python", f"{PACK}/blender/web_hero/render_panorama.py", "--",
           str(SAMPLES), "8192", f"preset={PRESET}", "near=800", f"out={OUT}/env"]
    t0 = time.time(); r = subprocess.run(cmd, cwd=PACK, env=env, capture_output=True, text=True)
    open(f"{OUT}/pano_run.log", "w").write(r.stdout[-100000:] + r.stderr[-20000:])
    print("panorama", round((time.time() - t0) / 60, 1), "min, rc", r.returncode); assert r.returncode == 0

In [ ]:
# Run record (it goes with the maps to assets/web_hero_v2_bake_<HHMM>)
import hashlib
st = json.load(open(f"{OUT}/maps/stages.json"))
files = sorted(glob.glob(f"{OUT}/**/*.*", recursive=True))
rec = {"hhmm": HHMM, "preset": PRESET, "only": ONLY or "all", "samples": SAMPLES, "stage_key": st.get("key"),
       "batch_seconds": {k: v["seconds"] for k, v in st["done"].items() if isinstance(v, dict) and "seconds" in v},
       "light_scales": {k: v.get("light_scale") for k, v in st["done"].items() if isinstance(v, dict) and "light_scale" in v},
       "gpu_memory_max_mib": gpu_mem_max,
       "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")[:2],
       "sha256": {os.path.relpath(f, OUT): hashlib.sha256(open(f, "rb").read()).hexdigest() for f in files if not f.endswith((".log", "run_record.json"))},
       "claim": "Internal only. Traffic is Visualization."}
json.dump(rec, open(f"{OUT}/run_record.json", "w"), indent=1)
print(json.dumps({k: v for k, v in rec.items() if k != "sha256"}, indent=1))